# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [18]:
df = pd.read_csv('data/AviationData.csv',encoding ='latin-1')

df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5772\3919368357.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv',encoding ='latin-1')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [19]:
pd.set_option('display.max_columns', None)
print(df.shape)
print(df.isnull().sum())
print(df['Aircraft.Category'].value_counts())
print(df['Amateur.Built'].value_counts())
print(df['FAR.Description'].value_counts())

(88889, 31)
Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [20]:
df['Event.Date'] =pd.to_datetime(df['Event.Date'], errors ='coerce')
df_filtered = df[
                  (df['Event.Date'].dt.year >= 1983)&
                  (df['Aircraft.Category'] == 'Airplane')&
                  (df['Amateur.Built'] == 'No')
                 ].copy()

print(df_filtered.shape)
df_filtered.head().T


(21447, 31)


,4149,4150,4171,4285,5957
Event.Id,20001214X42478,20001214X42478,20001214X42331,20001214X42672,20001214X44248
Investigation.Type,Incident,Incident,Accident,Accident,Incident
Accident.Number,LAX83IA149B,LAX83IA149A,ATL83FA140,FTW83LA177,MIA83IA210
Event.Date,1983-03-18 00:00:00,1983-03-18 00:00:00,1983-03-20 00:00:00,1983-04-02 00:00:00,1983-08-21 00:00:00
Location,"LOS ANGELES, CA","LOS ANGELES, CA","CROSSVILLE, TN","MCKINNEY, TX","NORFOLK, VA"
Country,United States,United States,United States,United States,United States
Latitude,NaN,NaN,NaN,NaN,NaN
Longitude,NaN,NaN,NaN,NaN,NaN
Airport.Code,LAX,LAX,NaN,TX05,NaN
Airport.Name,LOS ANGELES INTL,LOS ANGELES INTL,NaN,AERO COUNTRY,NaN


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [21]:
#  Missing injury values are filled with 0.
# If no injuries were recorded, it is reasonable to assume none occurred.

In [22]:

cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
        'Total.Minor.Injuries', 'Total.Uninjured']
for col in cols:
    df_filtered[col] = df_filtered[col].fillna(0).astype(int)
print(df_filtered[cols].describe())

df_filtered['Total.On.Board'] = (
    df_filtered['Total.Fatal.Injuries'] +
    df_filtered['Total.Serious.Injuries'] +
    df_filtered['Total.Minor.Injuries'] +
    df_filtered['Total.Uninjured']
)

df_filtered['Injury.Rate'] = df_filtered.apply(lambda row : (row['Total.Fatal.Injuries'] + row['Total.Serious.Injuries'])/row['Total.On.Board']if row['Total.On.Board'] > 0 else 0 ,axis = 1)
df_filtered[['Total.On.Board', 'Injury.Rate']].describe()

       Total.Fatal.Injuries  Total.Serious.Injuries  Total.Minor.Injuries  \
count          21447.000000            21447.000000          21447.000000   
mean               0.645871                0.280692              0.203245   
std                6.326507                2.218737              1.647516   
min                0.000000                0.000000              0.000000   
25%                0.000000                0.000000              0.000000   
50%                0.000000                0.000000              0.000000   
75%                0.000000                0.000000              0.000000   
max              295.000000              161.000000            200.000000   

       Total.Uninjured  
count     21447.000000  
mean          7.462489  
std          35.064707  
min           0.000000  
25%           0.000000  
50%           1.000000  
75%           2.000000  
max         588.000000  


,Total.On.Board,Injury.Rate
count,21447.000000,21447.000000
mean,8.592297,0.272002
std,35.910084,0.426343
min,0.000000,0.000000
25%,1.000000,0.000000
50%,2.000000,0.000000
75%,2.000000,0.666667
max,588.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [23]:
# Missing damage values are filled with 'Unknown' since we have no information to determine the extent of damage.
# Is_Destroyed is a binary column: 1 = destroyed, 0 = not destroyed

In [24]:
print(df_filtered['Aircraft.damage'].value_counts())
print('Missing:', df_filtered['Aircraft.damage'].isnull().sum())
df_filtered['Aircraft.damage'] = df_filtered['Aircraft.damage'].fillna('Unknown')
df_filtered['Is_Destroyed'] = (df_filtered['Aircraft.damage'] == 'Destroyed').astype(int)
print(df_filtered['Aircraft.damage'].value_counts())
print()
print(df_filtered['Is_Destroyed'].value_counts())



Aircraft.damage
Substantial    16990
Destroyed       2316
Minor            817
Unknown           97
Name: count, dtype: int64
Missing: 1227
Aircraft.damage
Substantial    16990
Destroyed       2316
Unknown         1324
Minor            817
Name: count, dtype: int64

Is_Destroyed
0    19131
1     2316
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [25]:
"""
Make Column Cleaning Tasks:
- Standardize capitalization to uppercase
- Removed  whitespace
- Merged duplicate names e.g CIRRUS DESIGN CORP -> CIRRUS
- Keept only makes with 50+ accidents for statistical robustness
"""


'\nMake Column Cleaning Tasks:\n- Standardize capitalization to uppercase\n- Remove  whitespace\n- Merge duplicate names e.g CIRRUS DESIGN CORP → CIRRUS\n- Keep only makes with 50+ accidents for statistical robustness\n'

In [26]:

df_filtered['Make'] = df_filtered['Make'].str.strip().str.upper()

df_filtered['Make'] = df_filtered['Make'].replace({
    'CIRRUS DESIGN CORP': 'CIRRUS',
    'AIR TRACTOR INC': 'AIR TRACTOR',
    'DEHAVILLAND': 'DE HAVILLAND',
    'BOMBARDIER INC': 'BOMBARDIER',
    'AVIAT AIRCRAFT INC': 'AVIAT'
})

df_filtered = df_filtered.dropna(subset=['Make'])
make_counts = df_filtered['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df_filtered = df_filtered[df_filtered['Make'].isin(valid_makes)]

print(f"remaining rows: {df_filtered.shape[0]}")
print(f"remaining unique makes: {df_filtered['Make'].nunique()}")
print()
print(df_filtered['Make'].value_counts())



Remaining rows: 17892
Remaining unique makes: 31

Make
CESSNA                            7146
PIPER                             3989
BEECH                             1431
BOEING                            1264
AIR TRACTOR                        425
MOONEY                             363
CIRRUS                             357
AIRBUS                             243
BELLANCA                           219
MAULE                              215
AERONCA                            200
DE HAVILLAND                       168
CHAMPION                           158
EMBRAER                            153
GRUMMAN                            147
AVIAT                              146
LUSCOMBE                           141
STINSON                            129
BOMBARDIER                         115
MCDONNELL DOUGLAS                  108
NORTH AMERICAN                     106
TAYLORCRAFT                         93
AERO COMMANDER                      90
SOCATA                              75
DIAMOND A

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [27]:
""" 
Model Column Cleaning Tasks:
- Droped missing model rows
- Standardize to uppercase
- Model names are NOT unique to each make (63 shared models found)
- Created Make_Model column as unique identifier
"""


' \nModel Column Cleaning Tasks:\n- Drop missing model rows\n- Standardize to uppercase\n- Model names are NOT unique to each make (63 shared models found)\n- Create Make_Model column as unique identifier\n'

In [28]:
print('Missing:', df_filtered['Model'].isnull().sum())
print()
print(df_filtered['Model'].value_counts().head(20))
print()
print('Total unique models:', df_filtered['Model'].nunique())
model_make_counts = df_filtered.groupby('Model')['Make'].nunique()
shared_models = model_make_counts[model_make_counts > 1]
print(f"Models shared across multiple makes: {len(shared_models)}")
print()
print(shared_models.head(20))
df_filtered = df_filtered.dropna(subset=['Model'])
df_filtered['Model'] = df_filtered['Model'].str.strip().str.upper()
df_filtered['Make_Model'] = df_filtered['Make'] + ' ' + df_filtered['Model']
print('Total unique Make_Model combinations:', df_filtered['Make_Model'].nunique())
print()
print(df_filtered['Make_Model'].value_counts().head(10))



Missing: 13

Model
172          769
737          403
152          316
182          304
172S         276
PA28         273
172N         249
SR22         240
180          213
A36          181
172M         180
150          179
PA-18-150    175
PA-28-140    169
172P         143
140          117
172R         109
170B         107
PA-28-180    105
PA-28-161    102
Name: count, dtype: int64

Total unique models: 2034
Models shared across multiple makes: 63

Model
100      2
112      2
112A     2
140      2
190      2
1900D    2
200      2
320      2
350      2
390      2
400      3
400A     2
401B     2
402A     2
402B     2
500      3
500 S    2
560      2
58       2
60       2
Name: Make, dtype: int64
Total unique Make_Model combinations: 2098

Make_Model
CESSNA 172     769
BOEING 737     403
CESSNA 152     316
CESSNA 182     304
CESSNA 172S    276
PIPER PA28     273
CESSNA 172N    249
CIRRUS SR22    240
CESSNA 180     213
CESSNA 172M    180
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [29]:
 """

After inspecting the following columns, I identified and executed these cleaning tasks:

Engine.Type
- 'UNK' and 'Unknown' mean the same thing — standardized to 'Unknown'

Weather.Condition
- 'Unk' and 'UNK' mean the same thing — standardized to 'Unknown'

Number.of.Engines
- 0.0 engines does not make sense for an airplane — replaced with NaN
- Missing values left as NaN since we cannot reliably impute engine count

Purpose.of.flight
- 'PUBS' and 'ASHO' are unclear  — standardized to 'Unknown'

Broad.phase.of.flight
- Values were already clean and consistent — no changes needed
 
  
"""


"\n\nAfter inspecting the following columns, we identified and executed these cleaning tasks:\n\nEngine.Type\n- 'UNK' and 'Unknown' mean the same thing — standardized to 'Unknown'\n\nWeather.Condition\n- 'Unk' and 'UNK' mean the same thing — standardized to 'Unknown'\n\nNumber.of.Engines\n- 0.0 engines does not make sense for an airplane — replaced with NaN\n- Missing values left as NaN since we cannot reliably impute engine count\n\nPurpose.of.flight\n- 'PUBS' and 'ASHO' are unclear  — standardized to 'Unknown'\n\nBroad.phase.of.flight\n- Values were already clean and consistent — no changes needed\n- This column has 86.3% missing values and will be dropped \n \n"

In [30]:
cols = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 
        'Purpose.of.flight', 'Broad.phase.of.flight']

for col in cols:
    print(f"Missing: {df_filtered[col].isnull().sum()}")
    print(df_filtered[col].value_counts())
    print()
df_filtered['Engine.Type'] = df_filtered['Engine.Type'].replace({'UNK':'Unknown'})
df_filtered['Weather.Condition'] = df_filtered['Weather.Condition'].replace({'Unk':'Unknown','UNK':'Unknown'})
df_filtered['Number.of.Engines'] =df_filtered['Number.of.Engines'].replace({0.0:None})
df_filtered['Purpose.of.flight'] = df_filtered['Purpose.of.flight'].replace({'PUBS':'Unknown','ASHO':'Unknown'})
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 
            'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"--- {col} --")
    print(df_filtered[col].value_counts())
    print()






Missing: 3214
Engine.Type
Reciprocating      12835
Turbo Prop           931
Turbo Fan            701
Unknown              105
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64

Missing: 2417
Weather.Condition
VMC    14295
IMC      905
Unk      186
UNK       76
Name: count, dtype: int64

Missing: 2089
Number.of.Engines
1.0    13222
2.0     2470
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64

Missing: 3047
Purpose.of.flight
Personal                     9844
Instructional                2410
Aerial Application            724
Business                      409
Unknown                       303
Positioning                   269
Skydiving                     157
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry                          72
Executive/corporate            65
Glider Tow                     29
Publ

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [31]:
"""
We drop columns where more than 50% of values are missing, 
as they are too insufficient to be useful for analysis.

Columns dropped:
- Schedule — 88.0% missing
- Broad.phase.of.flight — 86.3% missing
- Air.carrier — 52.7% missing

All remaining columns have less than 50% missing values 
and are retained for analysis.
"""


'\nWe drop columns where more than 50% of values are missing, \nas they are too insufficient to be useful for analysis.\n\nColumns dropped:\n- Schedule — 88.0% missing\n- Broad.phase.of.flight — 86.3% missing\n- Air.carrier — 52.7% missing\n\nAll remaining columns have less than 50% missing values \nand are retained for analysis.\n'

In [32]:

missing_pct = (df_filtered.isnull().sum() / len(df_filtered) * 100).sort_values(ascending=False)
print(missing_pct.round(1))
cols_to_drop = ['Schedule','Broad.phase.of.flight','Air.carrier']
df_filtered = df_filtered.drop(columns = cols_to_drop)
print(f"Remaining columns: {df_filtered.shape[1]}")
print(df_filtered.columns.tolist())







Schedule                  88.0
Broad.phase.of.flight     86.3
Air.carrier               52.7
Airport.Code              34.9
Airport.Name              34.3
Report.Status             21.2
Engine.Type               18.0
Purpose.of.flight         17.0
Weather.Condition         13.5
Number.of.Engines         11.7
Longitude                 10.6
Latitude                  10.6
Publication.Date           4.4
Injury.Severity            4.0
FAR.Description            1.9
Registration.Number        0.9
Location                   0.0
Country                    0.0
Total.Serious.Injuries     0.0
Total.On.Board             0.0
Total.Uninjured            0.0
Injury.Rate                0.0
Is_Destroyed               0.0
Total.Minor.Injuries       0.0
Event.Id                   0.0
Total.Fatal.Injuries       0.0
Investigation.Type         0.0
Amateur.Built              0.0
Model                      0.0
Make                       0.0
Aircraft.Category          0.0
Aircraft.damage            0.0
Event.Da

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [33]:
df_filtered.to_csv('data/AviationData_Cleaned.csv', index=False)
